# Zac Stritch-Hoddle PCA Analysis of OpenPose Data.

## Imports and helper functions.

In [8]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

def load_data(file_path, norm=None, detren=True, interpolate=True):
    """
    Load time-series pose data from a CSV file.
    The file should have column names for each keypoint (e.g., nose_x, nose_y).
    If norm is zscore, the data will be zscore normalized.
    if norm is unit, the data will be 0-1 normalized.
    """
    # load data
    data = pd.read_csv(file_path)

    # interpolate missing values
    if interpolate:
        data = interpolate_nans(data)

    # linear detrend
    if detren:
        data = linear_detrend(data)

    # normalize data
    if norm == 'zscore':
        return normalize_data(data, norm='zscore')
    elif norm == 'uint':
        return normalize_data(data, norm='uint')
    else:
        return data
    
def normalize_data(data, norm=None):
    """
    Normalize each column of data using z-score or min-max scaling.

    Parameters:
    - data (pd.DataFrame): The data to normalize.
    - norm (str): The normalization method. Options:
        - 'zscore': Normalize using z-score ((x - mean) / std).
        - 'unit': Normalize to the range [0, 1] ((x - min) / (max - min)).
        - None: No normalization.

    Returns:
    - pd.DataFrame: The normalized data.
    """
    if norm == 'zscore':
        return (data - data.mean()) / data.std()
    elif norm == 'uint':  # Corrected typo from "uint" to "unit"
        return (data - data.min()) / (data.max() - data.min())
    return data

def linear_detrend(data):
    """
    Linearly detrend each column of data.

    Parameters:
    - data (pd.DataFrame): The data to detrend.

    Returns:
    - pd.DataFrame: The detrended data.
    """
    return data.apply(lambda x: x - np.polyval(np.polyfit(data.index, x, 1), data.index))

def interpolate_nans(data):
    """
    Linearly interpolate NaN values in a DataFrame or Series.

    Parameters:
    - data (pd.DataFrame or pd.Series): The data with potential NaN values.

    Returns:
    - pd.DataFrame or pd.Series: Data with NaN values linearly interpolated.
    """
    if isinstance(data, pd.DataFrame):
        return data.interpolate(method='linear', axis=0, limit_direction='both')
    elif isinstance(data, pd.Series):
        return data.interpolate(method='linear', limit_direction='both')
    else:
        raise TypeError("Input must be a pandas DataFrame or Series.")
    
def perform_pca(data, selected_keypoints, n_components=4):
    """
    Perform PCA on the selected keypoints.
    """
    selected_data = data[selected_keypoints]
    pca = PCA(n_components=n_components)
    principal_components = pca.fit_transform(selected_data)

    return pca, principal_components

def cross_correlation(data, pca_ts, selected_keypoints):
    """
    Computes the cross-correlation between the PCA time series (PC1, PC2)
    and the original data time series for the selected keypoints.

    Parameters:
    - data (pd.DataFrame): Original time series data for the keypoints.
    - pca_ts (np.ndarray): Principal component time series (2D array from PCA).
    - selected_keypoints (list): List of keypoints to compare with PCA time series.

    Returns:
    - correlation_results (pd.DataFrame): DataFrame with correlation values for PC1 and PC2.
    """
    # Check that all selected keypoints exist in the data
    missing_keypoints = [key for key in selected_keypoints if key not in data.columns]
    if missing_keypoints:
        raise KeyError(f"The following keypoints are missing in the data: {missing_keypoints}")

    # Initialize a dictionary to store correlations
    correlation_results = {"Keypoint": [], "PC1_Correlation": [], "PC2_Correlation": []}

    # Iterate through each keypoint
    for keypoint in selected_keypoints:
        try:
            # Cross-correlation with PC1
            correlation_pc1 = np.corrcoef(data[keypoint], pca_ts[:, 0])[0, 1]

            # Cross-correlation with PC2
            correlation_pc2 = np.corrcoef(data[keypoint], pca_ts[:, 1])[0, 1]

            # Append results
            correlation_results["Keypoint"].append(keypoint)
            correlation_results["PC1_Correlation"].append(correlation_pc1)
            correlation_results["PC2_Correlation"].append(correlation_pc2)
        except Exception as e:
            print(f"Error processing keypoint {keypoint}: {e}")
            correlation_results["Keypoint"].append(keypoint)
            correlation_results["PC1_Correlation"].append(None)
            correlation_results["PC2_Correlation"].append(None)

    # Convert to DataFrame for better readability
    correlation_results_df = pd.DataFrame(correlation_results)
    return correlation_results_df

def plot_time_series(data, selected_keypoints, principal_components, pnum_components=3, title="Time-Series Data"):
    """
    Plot the time-series data and the first two principal components.
    """
    time = np.arange(data.shape[0])

    # Plot the time series of the selected keypoints
    plt.figure(figsize=(10, 5))
    for kp in selected_keypoints:
        plt.plot(time, data[kp], label=f"{kp}", linestyle=":", alpha=0.4)

    # Plot the time series of the first and second principal components
    for i in range(pnum_components):
        plt.plot(time, principal_components[:, i], label=f"PC{i+1}", linestyle="-")

    plt.xlabel("Time")
    plt.ylabel("Value")
    plt.title(title)
    plt.legend(["PC1", "PC2", "PC3"])  
    plt.grid(True)
    plt.show()

def plot_acc_time_series(principal_components, pnum_components=3, title="PCA ACC Data"):
    """
    Plot the time-series data and the first two principal components.
    """
    time = np.arange(principal_components.shape[0])

    # Plot the time series of the selected keypoints
    plt.figure(figsize=(10, 5))

    # convert to acceleration
    acceleration = np.diff(np.diff(principal_components, axis=0, prepend=0), axis=0, prepend=0)

    # Plot the time series of the first and second principal components
    for i in range(pnum_components):
        plt.plot(time, acceleration[:, i], label=f"PC{i+1}", linestyle="-")

    plt.xlabel("Time")
    plt.ylabel("ACC")
    plt.title(title)
    plt.legend(["PC1", "PC2", "PC3"])  
    plt.grid(True)
    plt.show()

## Load and set keypoint lists for analysis.

In [9]:
def extract_keypoints(file_path_or_data, sets=["face"]):
    """
    Extract keypoints from the dataset based on the specified sets.

    Parameters:
    - file_path_or_data (str or pd.DataFrame): Path to the CSV file or a DataFrame.
    - sets (list): List of sets to include in the output.

    Returns:
    - dict: Dictionary with keys as set names and values as lists of relevant column headers.
    """
    # Load the data if a file path is provided
    if isinstance(file_path_or_data, str):
        data = pd.read_csv(file_path_or_data)
    elif isinstance(file_path_or_data, pd.DataFrame):
        data = file_path_or_data
    else:
        raise ValueError("Input must be a file path or a pandas DataFrame.")

    # Get headers
    headers = data.columns.tolist()

    # Generate face keypoints (grouping everything as "face")
    face_keypoints = [
        header for header in headers if any(
            face_label in header.lower() for face_label in [
                "face", "eye", "pupil", "magnitude"
            ]
        )
    ]

    # Create a dictionary to store results
    keypoints_dict = {}
    if "face" in sets:
        keypoints_dict["face"] = face_keypoints

    return keypoints_dict

## Create folders for each participant to save plots.

In [6]:
import os

# Define the source and destination directories
source_directory = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Exp3_FaceEyeData'
destination_directory = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/PCA'

# Step 1: Get all the files in the source directory
files = os.listdir(source_directory)

# Step 2: Extract the unique first parts of the file names
unique_identifiers = set()

for file in files:
    if "_" in file:  # Ensure the file name contains the delimiter
        identifier = file.split("_")[0]
        unique_identifiers.add(identifier)

# Step 3: Create folders in the destination directory using the unique list
for identifier in unique_identifiers:
    folder_path = os.path.join(destination_directory, identifier)
    os.makedirs(folder_path, exist_ok=True)

print(f"Created {len(unique_identifiers)} folders in {destination_directory}")

Created 49 folders in /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/PCA


## Run and plot PCA analysis.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define paths
data_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Exp3_FaceEyeData'
output_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA'
output_csv = os.path.join(output_path, 'exp3_PCA_data.csv')

# Set normalization type
norm = None  # None, 'zscore', or 'uint'
do_plots = True

# Initialize a list to store stats for all trials
all_trial_stats = []

# Iterate through files in the directory
for file_name in os.listdir(data_path):
    if file_name.endswith('.csv') and "_" in file_name:
        # Extract participantID, trialNumber, and difficultyCondition
        parts = file_name.split("_")
        participantID = parts[0]
        trialNumber = parts[1]
        difficultyCondition = parts[2].split(".")[0]  # Remove the file extension

        # Locate the participant folder (assumes folders are already created)
        participant_folder = os.path.join(output_path, participantID)
        if not os.path.exists(participant_folder):
            print(f"Warning: Folder for participant {participantID} does not exist. Skipping.")
            continue

        # Construct file path
        file_path = os.path.join(data_path, file_name)

        # Load the data
        try:
            pose_data = load_data(file_path, norm=norm)  # Ensure load_data function is defined
        except FileNotFoundError:
            print(f"File not found: {file_path}")
            continue

        # Extract keypoints
        keypoints = extract_keypoints(pose_data, sets=["face"])  # Ensure extract_keypoints is defined
        keypoints = [key for sublist in keypoints.values() for key in sublist]

        # Perform PCA
        pca, principal_components = perform_pca(pose_data, keypoints, n_components=6)

        # Normalize principal components
        principal_components = normalize_data(principal_components, norm=norm)  # Ensure normalize_data is defined

        # Plot and save graphs
        if do_plots:
            # Explained Variance Ratio plot
            plt.figure()
            plt.plot(pca.explained_variance_ratio_, label='PCA')
            plt.xlabel('Principal Component')
            plt.ylabel('Explained Variance Ratio')
            plt.title(f"{participantID}_{trialNumber}_{difficultyCondition} Explained Variance Ratio")
            plt.legend()
            plt_path = os.path.join(participant_folder, f"{participantID}_{trialNumber}_{difficultyCondition}_Explained_Variance_Ratio.png")
            plt.savefig(plt_path)
            plt.close()

            # Pose Data PCA plot
            fig_pca = plt.figure()
            plt.plot(principal_components)
            plt.title(f"{participantID}_{trialNumber}_{difficultyCondition} Pose Data PCA")
            plt.xlabel("Time")
            plt.ylabel("Principal Components")
            plt_path = os.path.join(participant_folder, f"{participantID}_{trialNumber}_{difficultyCondition}_Pose_Data_PCA.png")
            fig_pca.savefig(plt_path)
            plt.close(fig_pca)

            # Pose Data PCA ACC plot
            fig_acc = plt.figure()
            plt.plot(principal_components)
            plt.title(f"{participantID}_{trialNumber}_{difficultyCondition} Pose Data PCA ACC")
            plt.xlabel("Time")
            plt.ylabel("ACC Components")
            plt_path = os.path.join(participant_folder, f"{participantID}_{trialNumber}_{difficultyCondition}_Pose_Data_PCA_ACC.png")
            fig_acc.savefig(plt_path)
            plt.close(fig_acc)

        # Calculate RMS
        rms = np.sqrt(np.mean(principal_components**2, axis=0))

        # Save trial stats
        trial_stats = {
            "participantID": participantID,
            "trialNumber": trialNumber,
            "difficultyCondition": difficultyCondition,
            "RMS": ','.join(map(str, rms)),  # Convert array to comma-separated string
            "PCA": ','.join(map(str, pca.explained_variance_ratio_)),  # Convert array to comma-separated string
        }
        all_trial_stats.append(trial_stats)

# Convert all trial stats to a DataFrame and save to CSV
df_stats = pd.DataFrame(all_trial_stats)
df_stats.to_csv(output_csv, index=False)

print(f"Analysis complete. Data saved to {output_csv}")

## Save first three PCA component time-series to a .csv file.

In [11]:
import os
import pandas as pd
import numpy as np

# Define paths
data_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/Exp3_FaceEyeData'
output_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA'
time_series_csv = os.path.join(output_path, 'exp3_PCA_time_series.csv')

# Set normalization type
norm = None  # None, 'zscore', or 'uint'

# Initialize a DataFrame to store PCA time-series data
pca_time_series = []

# Iterate through files in the directory
for file_name in os.listdir(data_path):
    if file_name.endswith('.csv') and "_" in file_name:
        # Extract participantID, trialNumber, and difficultyCondition
        parts = file_name.split("_")
        participantID = parts[0]
        trialNumber = parts[1]
        difficultyCondition = parts[2].split(".")[0]  # Remove the file extension

        # Construct file path
        file_path = os.path.join(data_path, file_name)

        # Load the data
        try:
            pose_data = load_data(file_path, norm=norm)  # Ensure load_data function is defined
        except FileNotFoundError:
            print(f"File not found: {file_path}")
            continue

        # Extract keypoints
        keypoints = extract_keypoints(pose_data, sets=["face"])  # Ensure extract_keypoints is defined
        keypoints = [key for sublist in keypoints.values() for key in sublist]

        # Perform PCA
        pca, principal_components = perform_pca(pose_data, keypoints, n_components=6)

        # Save each time point as a row
        for t, (pc1, pc2, pc3) in enumerate(principal_components[:, :3]):
            pca_time_series.append({
                "participantID": participantID,
                "trialNumber": trialNumber,
                "difficultyCondition": difficultyCondition,
                "timePoint": t,
                "PC1": pc1,
                "PC2": pc2,
                "PC3": pc3
            })

# Write PCA time-series to a .csv file
time_series_df = pd.DataFrame(pca_time_series)
time_series_df.to_csv(time_series_csv, index=False)

print(f"PCA time-series data saved to {time_series_csv}")

PCA time-series data saved to /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/exp3_PCA_time_series.csv


## Plotting AMI for first principal component time-series.

In [3]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Add the Utilities folder to the Python path
sys.path.append('/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/utils')

from ami_utils import ami

# Define paths
data_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA'
time_series_csv = os.path.join(data_path, 'exp3_PCA_time_series.csv')

# Load the PCA time-series data
data = pd.read_csv(time_series_csv)

# Iterate through all participants
for participant_id in data['participantID'].unique():
    participant_data = data[data['participantID'] == participant_id]

    # Iterate through unique trial numbers for the participant
    for trial_number in participant_data['trialNumber'].unique():
        trial_data = participant_data[participant_data['trialNumber'] == trial_number]

        # Get the difficulty condition
        difficulty_condition = trial_data['difficultyCondition'].iloc[0]

        # Extract the first principal component (PC1)
        pc1_time_series = trial_data['PC1'].values.astype(float)

        # Compute AMI for PC1
        ami_values = ami(pc1_time_series, 0, 200)  # Adjust lag range as needed

        # Participant folder path
        participant_folder = os.path.join(data_path, str(participant_id))

        # Ensure the folder exists
        if not os.path.exists(participant_folder):
            os.makedirs(participant_folder)

        # Save the AMI plot with appropriate labeling
        ami_plot_name = f'{participant_id}_{str(trial_number).zfill(2)}_{difficulty_condition}_ami_plot.png'
        ami_plot_path = os.path.join(participant_folder, ami_plot_name)

        # Plot and save the AMI
        plt.figure()
        plt.plot(ami_values[:, 0], ami_values[:, 1], 'b-')
        plt.xlabel('Lag')
        plt.ylabel('AMI')
        plt.title(f'AMI Analysis: {participant_id}_{str(trial_number).zfill(2)}_{difficulty_condition}')
        plt.savefig(ami_plot_path)
        plt.close()

        print(f'AMI plot saved for participant {participant_id}, trial {trial_number}, condition {difficulty_condition} at {ami_plot_path}')

print('AMI analysis and plotting completed for all participants!')

Processing AMI: 100%|██████████| 201/201 [00:04<00:00, 49.94it/s]


AMI plot saved for participant 3236, trial 1, condition L3 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3236/3236_01_L3_ami_plot.png


Processing AMI: 100%|██████████| 201/201 [00:04<00:00, 50.05it/s]


AMI plot saved for participant 3236, trial 3, condition M2 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3236/3236_03_M2_ami_plot.png


Processing AMI: 100%|██████████| 201/201 [00:04<00:00, 49.62it/s]


...

AMI plot saved for participant 3226, trial 3, condition H2 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3226/3226_03_H2_ami_plot.png


Processing AMI: 100%|██████████| 201/201 [00:04<00:00, 49.85it/s]


AMI plot saved for participant 3226, trial 2, condition M2 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3226/3226_02_M2_ami_plot.png


Processing AMI:  13%|█▎        | 26/201 [00:00<00:03, 49.62it/s]


KeyboardInterrupt: 

## Performing a test FNN analysis on one participant's data.

In [5]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Add the Utilities folder to the Python path
sys.path.append('/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Exp3 OpenPose Data/utils')

from fnn_utils import fnn

# Define paths
data_path = '/Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA'
time_series_csv = os.path.join(data_path, 'exp3_PCA_time_series.csv')

# Load the PCA time-series data
data = pd.read_csv(time_series_csv)

# Iterate through all participants
for participant_id in data['participantID'].unique():
    participant_data = data[data['participantID'] == participant_id]

    # Iterate through unique trial numbers for the participant
    for trial_number in participant_data['trialNumber'].unique():
        trial_data = participant_data[participant_data['trialNumber'] == trial_number]

        # Get the difficulty condition
        difficulty_condition = trial_data['difficultyCondition'].iloc[0]

        # Extract the first principal component (PC1)
        pc1_time_series = trial_data['PC1'].values.astype(float)

        # Compute FNN for PC1
        fnn_ds, fnn_percent = fnn(pc1_time_series, 15, 1, 10)

        # Participant folder path
        participant_folder = os.path.join(data_path, str(participant_id))

        # Ensure the folder exists
        if not os.path.exists(participant_folder):
            os.makedirs(participant_folder)

        # Save the FNN plot with appropriate labeling
        fnn_plot_name = f'{participant_id}_{str(trial_number).zfill(2)}_{difficulty_condition}_fnn_plot.png'
        fnn_plot_path = os.path.join(participant_folder, fnn_plot_name)

        # Plot and save the FNN
        plt.figure()
        plt.plot(fnn_ds, fnn_percent, 'k-o')
        plt.xlim([fnn_ds[0], fnn_ds[-1]])
        plt.xlabel('# Embedding Dimensions')
        plt.ylabel('% False Nearest Neighbors')
        plt.title(f'FNN Analysis: {participant_id}_{str(trial_number).zfill(2)}_{difficulty_condition}')
        plt.savefig(fnn_plot_path)
        plt.close()

        print(f'FNN plot saved for participant {participant_id}, trial {trial_number}, condition {difficulty_condition} at {fnn_plot_path}')

print('FNN analysis and plotting completed for all participants!')

Processing FNN: 100%|██████████| 10/10 [00:03<00:00,  2.95it/s]


FNN plot saved for participant 3236, trial 1, condition L3 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3236/3236_01_L3_fnn_plot.png


Processing FNN: 100%|██████████| 10/10 [00:03<00:00,  2.94it/s]


FNN plot saved for participant 3236, trial 3, condition M2 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3236/3236_03_M2_fnn_plot.png


Processing FNN: 100%|██████████| 10/10 [00:03<00:00,  2.89it/s]


...

FNN plot saved for participant 3233, trial 2, condition H2 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3233/3233_02_H2_fnn_plot.png


Processing FNN: 100%|██████████| 10/10 [00:03<00:00,  2.89it/s]


FNN plot saved for participant 3233, trial 3, condition M2 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3233/3233_03_M2_fnn_plot.png


Processing FNN: 100%|██████████| 10/10 [00:03<00:00,  2.91it/s]

FNN plot saved for participant 3233, trial 1, condition L3 at /Users/zacstritch/Desktop/Research Assisting/RQA/Zac Analysis/Data Sets/Experiment 3 PCA/3233/3233_01_L3_fnn_plot.png
FNN analysis and plotting completed for all participants!
